In [3]:
# ============================================================
# TASK 3: Qualitative Analysis
# Realism, Failure Modes & Representative Samples
# ============================================================
# This notebook loads all three trained models and performs
# a deep qualitative analysis of generated names including:
#   - Realism scoring using linguistic heuristics
#   - Failure mode detection and categorisation
#   - Representative sample tables per model
#   - Comparative summary report
# ============================================================

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import random
import re
from collections import Counter
from google.colab import files

# ============================================================
# STEP 0: Upload Files
# ============================================================

print("Upload TrainingNames.txt and all three .pt model files...")
uploaded = files.upload()

with open("TrainingNames.txt", "r", encoding="utf-8") as f:
    training_names = set(line.strip().lower() for line in f if line.strip())

print(f"Training set loaded: {len(training_names)} names")


# ============================================================
# STEP 1: Rebuild Vocabulary (identical to Task 1 & 2)
# ============================================================

all_chars      = sorted(set("".join(training_names)))
special_tokens = ["<PAD>", "<SOS>", "<EOS>"]
vocab          = special_tokens + all_chars
char2idx       = {ch: idx for idx, ch in enumerate(vocab)}
idx2char       = {idx: ch  for idx, ch in enumerate(vocab)}

PAD_IDX    = char2idx["<PAD>"]
SOS_IDX    = char2idx["<SOS>"]
EOS_IDX    = char2idx["<EOS>"]
VOCAB_SIZE = len(vocab)

print(f"Vocabulary size: {VOCAB_SIZE}")


# ============================================================
# STEP 2: Rebuild Model Architectures
# ============================================================

class VanillaRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim=64,
                 hidden_size=256, num_layers=2, dropout=0.3):
        super(VanillaRNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers
        self.embedding   = nn.Embedding(vocab_size, embedding_dim,
                                        padding_idx=PAD_IDX)
        self.rnn = nn.RNN(
            input_size=embedding_dim, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            nonlinearity='tanh'
        )
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden=None):
        embedded       = self.dropout(self.embedding(x))
        output, hidden = self.rnn(embedded, hidden)
        return self.fc(self.dropout(output)), hidden

    def init_hidden(self, batch_size, device):
        return torch.zeros(self.num_layers, batch_size,
                           self.hidden_size, device=device)


class BiLSTMEncoderDecoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim=64,
                 hidden_size=256, num_layers=2, dropout=0.3):
        super(BiLSTMEncoderDecoder, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers
        self.embedding   = nn.Embedding(vocab_size, embedding_dim,
                                        padding_idx=PAD_IDX)
        self.dropout     = nn.Dropout(dropout)
        self.encoder = nn.LSTM(
            input_size=embedding_dim, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )
        # Project concatenated fwd+bwd hidden/cell (2H) down to H
        self.h_proj = nn.Linear(hidden_size * 2, hidden_size)
        self.c_proj = nn.Linear(hidden_size * 2, hidden_size)
        self.decoder = nn.LSTM(
            input_size=embedding_dim, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.fc = nn.Linear(hidden_size, vocab_size)

    def _encode_sos(self, device):
        """Encode [SOS] to get warm decoder initial state."""
        sos       = torch.tensor([[SOS_IDX]], dtype=torch.long, device=device)
        emb       = self.dropout(self.embedding(sos))
        _, (h, c) = self.encoder(emb)
        B         = 1
        h = h.view(self.num_layers, 2, B, self.hidden_size)
        c = c.view(self.num_layers, 2, B, self.hidden_size)
        h = torch.tanh(self.h_proj(
                torch.cat([h[:, 0, :, :], h[:, 1, :, :]], dim=2)))
        c = torch.tanh(self.c_proj(
                torch.cat([c[:, 0, :, :], c[:, 1, :, :]], dim=2)))
        return h, c

    def forward(self, x, hidden=None):
        device      = x.device
        B           = x.size(0)
        h, c        = self._encode_sos(device)
        h           = h.expand(-1, B, -1).contiguous()
        c           = c.expand(-1, B, -1).contiguous()
        emb         = self.dropout(self.embedding(x))
        out, hidden = self.decoder(emb, (h, c))
        return self.fc(self.dropout(out)), hidden

    def init_hidden(self, batch_size, device):
        h = torch.zeros(self.num_layers, batch_size,
                        self.hidden_size, device=device)
        c = torch.zeros(self.num_layers, batch_size,
                        self.hidden_size, device=device)
        return (h, c)


class Attention(nn.Module):
    def __init__(self, hidden_size):
        super(Attention, self).__init__()
        self.attn = nn.Linear(hidden_size * 2, hidden_size)
        self.v    = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        seq_len         = encoder_outputs.size(1)
        hidden_expanded = hidden.unsqueeze(1).repeat(1, seq_len, 1)
        energy          = torch.tanh(
            self.attn(torch.cat([hidden_expanded, encoder_outputs], dim=2))
        )
        attn_weights = torch.softmax(self.v(energy).squeeze(2), dim=1)
        context      = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs)
        return context.squeeze(1), attn_weights


class RNNWithAttention(nn.Module):
    def __init__(self, vocab_size, embedding_dim=64,
                 hidden_size=256, num_layers=2, dropout=0.3):
        super(RNNWithAttention, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers
        self.embedding   = nn.Embedding(vocab_size, embedding_dim,
                                        padding_idx=PAD_IDX)
        self.rnn = nn.RNN(
            input_size=embedding_dim, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            nonlinearity='tanh'
        )
        self.attention = Attention(hidden_size)
        self.dropout   = nn.Dropout(dropout)
        self.fc        = nn.Linear(hidden_size * 2, vocab_size)

    def forward(self, x, hidden=None):
        emb             = self.dropout(self.embedding(x))
        rnn_out, hidden = self.rnn(emb, hidden)
        T      = rnn_out.size(1)
        logits = []
        for t in range(T):
            ctx, _ = self.attention(hidden[-1], rnn_out[:, :t+1, :])
            step   = torch.cat([rnn_out[:, t, :], ctx], dim=1)
            logits.append(self.fc(self.dropout(step)))
        return torch.stack(logits, dim=1), hidden

    def init_hidden(self, batch_size, device):
        return torch.zeros(self.num_layers, batch_size,
                           self.hidden_size, device=device)

    def decode_step(self, x, hidden, history):
        """Single inference step — grows history by 1 each call."""
        emb             = self.dropout(self.embedding(x))
        rnn_out, hidden = self.rnn(emb, hidden)
        history         = torch.cat([history, rnn_out], dim=1)
        ctx, _          = self.attention(hidden[-1], history)
        combined        = torch.cat([rnn_out.squeeze(1), ctx], dim=1)
        logits          = self.fc(self.dropout(combined))
        return logits, hidden, history


# ============================================================
# STEP 3: Load Model Weights
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {device}")

rnn_model  = VanillaRNN(VOCAB_SIZE).to(device)
lstm_model = BiLSTMEncoderDecoder(VOCAB_SIZE).to(device)
attn_model = RNNWithAttention(VOCAB_SIZE).to(device)

rnn_model.load_state_dict(torch.load("vanilla_rnn.pt",    map_location=device))
lstm_model.load_state_dict(torch.load("blstm.pt",         map_location=device))
attn_model.load_state_dict(torch.load("rnn_attention.pt", map_location=device))

rnn_model.eval()
lstm_model.eval()
attn_model.eval()

print("All models loaded and set to eval mode.")


# ============================================================
# STEP 4: Name Generation
# ============================================================

def generate_name(model, device, max_len=20, temperature=0.8):
    """Single name generation via autoregressive character sampling."""
    with torch.no_grad():
        input_char = torch.tensor([[SOS_IDX]], dtype=torch.long).to(device)
        hidden     = model.init_hidden(batch_size=1, device=device)
        generated  = []

        for _ in range(max_len):
            logits, hidden = model(input_char, hidden)
            probs          = torch.softmax(logits[0, 0] / temperature, dim=0)
            next_idx       = torch.multinomial(probs, num_samples=1).item()

            if next_idx == EOS_IDX:
                break
            if next_idx not in (PAD_IDX, SOS_IDX):
                generated.append(idx2char[next_idx])

            input_char = torch.tensor([[next_idx]], dtype=torch.long).to(device)

    return "".join(generated).strip()


def generate_name_bilstm(model, device, max_len=20, temperature=0.8):
    """Single name generation via autoregressive character sampling."""
    with torch.no_grad():
        h, c       = model._encode_sos(device)
        input_char = torch.tensor([[SOS_IDX]], dtype=torch.long).to(device)
        generated  = []

        for _ in range(max_len):
            emb         = model.dropout(model.embedding(input_char))
            out, (h, c) = model.decoder(emb, (h, c))
            probs       = torch.softmax(model.fc(out[0, 0]) / temperature,
                                        dim=0)
            next_idx    = torch.multinomial(probs, num_samples=1).item()

            if next_idx == EOS_IDX:
                break
            if next_idx not in (PAD_IDX, SOS_IDX):
                generated.append(idx2char[next_idx])

            input_char = torch.tensor([[next_idx]], dtype=torch.long).to(device)

    return "".join(generated).strip()


def generate_name_attn(model, device, max_len=20, temperature=0.8):
    """Single name generation via autoregressive character sampling."""
    with torch.no_grad():
        input_char = torch.tensor([[SOS_IDX]], dtype=torch.long).to(device)
        hidden     = model.init_hidden(batch_size=1, device=device)
        generated  = []

        # Bootstrap: seed history with the real SOS RNN output
        emb             = model.dropout(model.embedding(input_char))
        rnn_out, hidden = model.rnn(emb, hidden)
        history         = rnn_out  # (1, 1, H)

        # First character prediction from SOS context
        ctx, _   = model.attention(hidden[-1], history)
        combined = torch.cat([rnn_out.squeeze(1), ctx], dim=1)
        logits   = model.fc(model.dropout(combined))
        probs    = torch.softmax(logits[0] / temperature, dim=0)
        next_idx = torch.multinomial(probs, num_samples=1).item()

        if next_idx == EOS_IDX:
            return ""
        if next_idx not in (PAD_IDX, SOS_IDX):
            generated.append(idx2char[next_idx])

        input_char = torch.tensor([[next_idx]], dtype=torch.long).to(device)

        for _ in range(max_len - 1):
            logits, hidden, history = model.decode_step(
                input_char, hidden, history)
            probs    = torch.softmax(logits[0] / temperature, dim=0)
            next_idx = torch.multinomial(probs, num_samples=1).item()

            if next_idx == EOS_IDX:
                break
            if next_idx not in (PAD_IDX, SOS_IDX):
                generated.append(idx2char[next_idx])

            input_char = torch.tensor([[next_idx]], dtype=torch.long).to(device)

    return "".join(generated).strip()


def generate_batch(model, device, n=300, temperature=0.8):
    """Generate n valid names (length > 1) from the model."""
    results = []
    # Keep generating until we collect n valid names
    while len(results) < n:
        if isinstance(model, BiLSTMEncoderDecoder):
            name = generate_name_bilstm(model, device, temperature=temperature)
        elif isinstance(model, RNNWithAttention):
            name = generate_name_attn(model, device, temperature=temperature)
        else:
            name = generate_name(model, device, temperature=temperature)
        if len(name) > 1:
            results.append(name.lower())
    return results


# ============================================================
# STEP 5: Realism Scoring Heuristics
# ============================================================
# We define realism using four linguistic rules derived from
# patterns observed in authentic Indian names:
#
#   Rule 1 — Length plausibility  : 3–12 characters
#   Rule 2 — No repeated triplets : "aaa", "rrr" etc. are noise
#   Rule 3 — Vowel presence       : every real name has at least one vowel
#   Rule 4 — Consonant cluster cap: no more than 3 consecutive consonants
#
# Each rule contributes 1 point; realism score = sum / 4 (0–1 scale).

VOWELS = set("aeiouáéíóú")

def is_length_plausible(name):
    """Indian names are almost always 3–12 characters long."""
    return 3 <= len(name) <= 12

def has_no_repeated_triplets(name):
    """
    Flags names with three identical consecutive characters (e.g. 'aaa').
    These are artefacts of the model getting stuck in a character loop.
    """
    for i in range(len(name) - 2):
        if name[i] == name[i+1] == name[i+2]:
            return False
    return True

def has_vowel(name):
    """Every legitimate name must contain at least one vowel."""
    return any(ch in VOWELS for ch in name)

def no_excessive_consonant_cluster(name):
    """
    More than 3 consecutive consonants is phonetically implausible
    for Indian names (e.g. 'krshtnv' would be flagged).
    """
    consonant_run = 0
    for ch in name:
        if ch not in VOWELS:
            consonant_run += 1
            if consonant_run > 3:
                return False
        else:
            consonant_run = 0
    return True

def realism_score(name):
    """
    Aggregates all four rules into a single score between 0.0 and 1.0.
    1.0 = passes all rules (most realistic)
    0.0 = fails all rules (clearly malformed)
    """
    checks = [
        is_length_plausible(name),
        has_no_repeated_triplets(name),
        has_vowel(name),
        no_excessive_consonant_cluster(name),
    ]
    return sum(checks) / len(checks)


# ============================================================
# STEP 6: Failure Mode Detection
# ============================================================
# We categorise each generated name into one of five failure
# modes (or mark it as VALID if none apply).

def detect_failure_mode(name):
    """
    Assigns a failure mode label to a generated name.

    Failure modes:
      TOO_SHORT     : 1–2 characters — model terminated too early
      TOO_LONG      : 13+ characters — model never learned to stop
      REPETITION    : three or more identical consecutive characters
      NO_VOWEL      : all consonants — phonetically unpronounceable
      CONSONANT_CLUSTER : 4+ consecutive consonants — unnatural cluster
      VALID         : passes all checks — a plausible name
    """
    if len(name) <= 2:
        return "TOO_SHORT"
    if len(name) > 12:
        return "TOO_LONG"
    if not has_no_repeated_triplets(name):
        return "REPETITION"
    if not has_vowel(name):
        return "NO_VOWEL"
    if not no_excessive_consonant_cluster(name):
        return "CONSONANT_CLUSTER"
    return "VALID"


def failure_mode_summary(names):
    """
    Counts how many names fall into each failure category.
    Returns a dict sorted by frequency (most common first).
    """
    modes  = [detect_failure_mode(n) for n in names]
    counts = Counter(modes)
    total  = len(names)
    # Convert to percentage breakdown
    return {mode: round(cnt / total * 100, 2)
            for mode, cnt in counts.most_common()}


# ============================================================
# STEP 7: Representative Sample Selection
# ============================================================

def get_representative_samples(names, n_valid=10, n_failed=5):
    """
    Selects a balanced set of representative examples:
      - n_valid  : names that passed all realism checks (VALID)
      - n_failed : one example per failure mode detected

    This gives an honest view of both good outputs and failure cases.
    """
    valid_names  = [n for n in names if detect_failure_mode(n) == "VALID"]
    failed_names = [n for n in names if detect_failure_mode(n) != "VALID"]

    # Sample without replacement where possible
    valid_sample  = random.sample(valid_names,
                                  min(n_valid, len(valid_names)))

    # Collect one example per unique failure type
    failure_examples = {}
    for name in failed_names:
        mode = detect_failure_mode(name)
        if mode not in failure_examples:
            failure_examples[mode] = name

    return valid_sample, failure_examples


# ============================================================
# STEP 8: Run Full Qualitative Analysis
# ============================================================

N_GENERATE  = 300
TEMPERATURE = 0.8

print(f"\nGenerating {N_GENERATE} names per model...\n")

models_info = [
    ("Vanilla RNN",        rnn_model),
    ("Bidirectional LSTM", lstm_model),
    ("RNN + Attention",    attn_model),
]

# Master results storage
qualitative_results = []
all_generated       = {}

for model_name, model in models_info:
    print(f"  [{model_name}] Generating...")
    generated = generate_batch(model, device,
                               n=N_GENERATE, temperature=TEMPERATURE)
    all_generated[model_name] = generated

    # --- Realism ---
    scores    = [realism_score(n) for n in generated]
    avg_score = np.mean(scores)
    high_real = sum(1 for s in scores if s == 1.0)   # perfect score count

    # --- Failure Modes ---
    fail_summary = failure_mode_summary(generated)
    valid_pct    = fail_summary.get("VALID", 0)

    # --- Samples ---
    valid_samples, fail_examples = get_representative_samples(generated)

    qualitative_results.append({
        "Model"              : model_name,
        "Avg Realism Score"  : round(avg_score, 4),
        "Perfect Realism %"  : round(high_real / N_GENERATE * 100, 2),
        "Valid Names %"      : valid_pct,
        "Failure Breakdown"  : fail_summary,
        "Valid Samples"      : valid_samples,
        "Failure Examples"   : fail_examples,
    })

    print(f"    Avg Realism  : {avg_score:.4f}")
    print(f"    Valid %      : {valid_pct}%")
    print(f"    Failures     : {fail_summary}\n")


# ============================================================
# STEP 9: Print Full Qualitative Report
# ============================================================

print("\n" + "="*70)
print("              QUALITATIVE ANALYSIS REPORT")
print("="*70)

# --- Summary Comparison Table ---
summary_rows = []
for r in qualitative_results:
    summary_rows.append({
        "Model"             : r["Model"],
        "Avg Realism Score" : r["Avg Realism Score"],
        "Perfect Realism %" : r["Perfect Realism %"],
        "Valid Names %"     : r["Valid Names %"],
    })

summary_df = pd.DataFrame(summary_rows)
print("\n  REALISM SUMMARY TABLE")
print("  " + "-"*55)
print(summary_df.to_string(index=False))
print("  " + "-"*55)

# --- Per-Model Detailed Analysis ---
for r in qualitative_results:
    print(f"\n{'='*70}")
    print(f"  MODEL: {r['Model']}")
    print(f"{'='*70}")

    # Realism metrics
    print(f"\n  Realism Metrics:")
    print(f"    Average realism score  : {r['Avg Realism Score']} / 1.0")
    print(f"    Names with full score  : {r['Perfect Realism %']}%")
    print(f"    Names passing all rules: {r['Valid Names %']}%")

    # Failure mode breakdown
    print(f"\n  Failure Mode Breakdown:")
    print(f"  {'Mode':<22} {'% of Generated'}")
    print(f"  {'-'*38}")
    for mode, pct in r["Failure Breakdown"].items():
        bar = "▓" * int(pct / 2)   # 1 block per 2%
        print(f"  {mode:<22} {pct:>6}%  {bar}")

    # Representative valid samples
    print(f"\n  Representative VALID Names (realistic outputs):")
    samples = r["Valid Samples"]
    for i in range(0, len(samples), 3):
        row = samples[i:i+3]
        print("    " + "   ".join(f"{n.capitalize():<15}" for n in row))

    # Representative failure examples
    print(f"\n  Representative FAILURE Examples (one per mode):")
    print(f"  {'Failure Mode':<22} {'Example':<20} {'Explanation'}")
    print(f"  {'-'*65}")
    failure_explanations = {
        "TOO_SHORT"          : "Model emitted EOS too early",
        "TOO_LONG"           : "Model failed to learn stopping condition",
        "REPETITION"         : "Hidden state got stuck in a character loop",
        "NO_VOWEL"           : "Model sampled only consonants consecutively",
        "CONSONANT_CLUSTER"  : "Unnatural phonetic cluster learned from noise",
    }
    for mode, example in r["Failure Examples"].items():
        explanation = failure_explanations.get(mode, "Unknown")
        print(f"  {mode:<22} {example.capitalize():<20} {explanation}")


# ============================================================
# STEP 10: Cross-Model Failure Comparison Table
# ============================================================

print(f"\n{'='*70}")
print("  CROSS-MODEL FAILURE MODE COMPARISON (%)")
print(f"{'='*70}")

# Collect all unique failure modes across models
all_modes = set()
for r in qualitative_results:
    all_modes.update(r["Failure Breakdown"].keys())
all_modes = sorted(all_modes)

# Build comparison dataframe
comparison_rows = {"Failure Mode": all_modes}
for r in qualitative_results:
    comparison_rows[r["Model"]] = [
        r["Failure Breakdown"].get(mode, 0.0) for mode in all_modes
    ]

comparison_df = pd.DataFrame(comparison_rows)
print(comparison_df.to_string(index=False))


# ============================================================
# STEP 11: Save and Download Report
# ============================================================

# Save summary table
summary_df.to_csv("qualitative_summary.csv", index=False)

# Save full failure comparison
comparison_df.to_csv("failure_mode_comparison.csv", index=False)

# Save all generated names with their labels for manual inspection
rows = []
for model_name, generated in all_generated.items():
    for name in generated:
        rows.append({
            "Model"         : model_name,
            "Name"          : name.capitalize(),
            "Realism Score" : realism_score(name),
            "Failure Mode"  : detect_failure_mode(name),
            "In Training"   : name in training_names,
        })

all_names_df = pd.DataFrame(rows)
all_names_df.to_csv("all_generated_names.csv", index=False)

print("\n\nFiles saved:")
print("  qualitative_summary.csv      — realism metrics per model")
print("  failure_mode_comparison.csv  — failure breakdown across models")
print("  all_generated_names.csv      — every generated name with labels")

files.download("qualitative_summary.csv")
files.download("failure_mode_comparison.csv")
files.download("all_generated_names.csv")

print("\nTask 3 complete.")

Upload TrainingNames.txt and all three .pt model files...


Saving blstm.pt to blstm.pt
Saving rnn_attention.pt to rnn_attention.pt
Saving vanilla_rnn.pt to vanilla_rnn.pt
Training set loaded: 1000 names
Vocabulary size: 29

Using device: cpu
All models loaded and set to eval mode.

Generating 300 names per model...

  [Vanilla RNN] Generating...
    Avg Realism  : 0.9983
    Valid %      : 99.33%
    Failures     : {'VALID': 99.33, 'CONSONANT_CLUSTER': 0.33, 'TOO_SHORT': 0.33}

  [Bidirectional LSTM] Generating...
    Avg Realism  : 0.9917
    Valid %      : 97.0%
    Failures     : {'VALID': 97.0, 'CONSONANT_CLUSTER': 1.33, 'TOO_SHORT': 1.0, 'TOO_LONG': 0.33, 'REPETITION': 0.33}

  [RNN + Attention] Generating...
    Avg Realism  : 0.9958
    Valid %      : 98.67%
    Failures     : {'VALID': 98.67, 'CONSONANT_CLUSTER': 0.67, 'TOO_SHORT': 0.33, 'TOO_LONG': 0.33}


              QUALITATIVE ANALYSIS REPORT

  REALISM SUMMARY TABLE
  -------------------------------------------------------
             Model  Avg Realism Score  Perfect Realism %

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Task 3 complete.
